# Week 8 — Evaluation Expansion
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 7  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3, models from Week 5, 6 & 7

**Goal:** Compute MAPE and MdAPE (not just R²) for all four models, then dig 
deeper into XGBoost's performance by price band to see which price ranges the 
model handles well vs. poorly.

## Setup
Loading the cleaned dataset and reusing the same window, test month, and 
model settings established in prior weeks, so this evaluation stays 
consistent with everything before it.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

BASE_FEATURE_COLS = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'
BEST_WINDOW = 3
test_month = pd.Period('2026-06', freq='M')

def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

def mdape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true))

print(f'Loaded {len(model_df):,} rows')

Loaded 411,419 rows


## 1. Rebuild Week 6/7 Feature Set
Same as notebook 6 -- rebuilding BedBathRatio and the school district join 
here so this notebook can run on its own without depending on other 
notebooks having been run first.

In [2]:
model_df['BedBathRatio'] = model_df['BedroomsTotal'] / model_df['BathroomsTotalInteger'].replace(0, np.nan)
model_df['BedBathRatio'] = model_df['BedBathRatio'].fillna(model_df['BedBathRatio'].median())

district_path = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\scripts\DistrictAreas2526_-284845464123469011.geojson'
districts = gpd.read_file(district_path)
districts_unified = districts[districts['DistrictType'] == 'Unified'].copy()

properties_gdf = gpd.GeoDataFrame(
    model_df,
    geometry=gpd.points_from_xy(model_df['Longitude'], model_df['Latitude']),
    crs='EPSG:4326'
).to_crs(districts_unified.crs)

joined = gpd.sjoin(properties_gdf, districts_unified[['DistrictName', 'geometry']], how='left', predicate='within')
model_df['SchoolDistrict'] = joined['DistrictName'].values
model_df['SchoolDistrict'] = model_df['SchoolDistrict'].fillna('No_Unified_District')
model_df = model_df[~model_df.index.duplicated(keep='first')]

train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)

train_dummies = pd.get_dummies(train_df['SchoolDistrict'], prefix='Dist', drop_first=True)
test_dummies = pd.get_dummies(test_df['SchoolDistrict'], prefix='Dist', drop_first=True)
train_dummies, test_dummies = train_dummies.align(test_dummies, join='left', axis=1, fill_value=0)

train_df = pd.concat([train_df.reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)

DISTRICT_COLS = train_dummies.columns.tolist()
FEATURE_COLS = BASE_FEATURE_COLS + ['BedBathRatio'] + DISTRICT_COLS

X_train, y_train = train_df[FEATURE_COLS], train_df[target_col]
X_test, y_test = test_df[FEATURE_COLS], test_df[target_col]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(FEATURE_COLS)}')

Train: 35,186 | Test: 12,841 | Features: 311


## 2. Train All Four Models
Retraining Linear Regression, Decision Tree, Random Forest, and XGBoost using 
each model's best settings found in Weeks 4-7, so all four can be compared 
side by side on the same final metrics.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LinearRegression().fit(X_train_scaled, y_train)
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)
rf_model = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(X_train, y_train)
xgb_model = XGBRegressor(max_depth=8, learning_rate=0.05, n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)

preds = {
    'Linear Regression': lr_model.predict(X_test_scaled),
    'Decision Tree': dt_model.predict(X_test),
    'Random Forest': rf_model.predict(X_test),
    'XGBoost': xgb_model.predict(X_test),
}
print('All 4 models trained.')

All 4 models trained.


## 3. Full Metrics: R², MAPE, MdAPE
MAPE is the *average* % error -- sensitive to a few very bad predictions 
dragging the number up. MdAPE is the *median* % error -- more robust, tells 
us how the "typical" prediction performs regardless of a handful of outliers.

Going beyond R² as required this week. MAPE (Mean Absolute Percentage Error) 
is the average % error across all predictions -- a few very bad misses can 
drag this number up. MdAPE (Median Absolute Percentage Error) is the middle 
value -- more robust to outlier misses, and tells us how a "typical" 
prediction performs.

In [4]:
metrics_summary = pd.DataFrame([
    {
        'model': name,
        'R2': r2_score(y_test, p),
        'MAPE': mean_absolute_percentage_error(y_test, p),
        'MdAPE': mdape(y_test.values, p)
    }
    for name, p in preds.items()
])
metrics_summary

,model,R2,MAPE,MdAPE
0,Linear Regression,0.649220,0.344517,0.221888
1,Decision Tree,0.774022,0.224016,0.148830
2,Random Forest,0.875719,0.151161,0.094961
3,XGBoost,0.892493,0.153588,0.101509


## 4. Price Band Analysis (XGBoost)
Splitting test properties into four price ranges and computing MAPE/MdAPE 
within each one, to check whether the model performs consistently across the 
price spectrum or struggles more at a particular end.

In [5]:
price_bins = [0, 500_000, 1_000_000, 2_000_000, np.inf]
price_labels = ['Under $500k', '$500k-$1M', '$1M-$2M', 'Over $2M']

band_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': preds['XGBoost']
})
band_df['price_band'] = pd.cut(band_df['actual'], bins=price_bins, labels=price_labels)

band_results = band_df.groupby('price_band', observed=True).apply(
    lambda g: pd.Series({
        'n_properties': len(g),
        'MAPE': mean_absolute_percentage_error(g['actual'], g['predicted']),
        'MdAPE': mdape(g['actual'].values, g['predicted'].values)
    })
).reset_index()

band_results

C:\Users\monik\AppData\Local\Temp\ipykernel_30956\2953395952.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  band_results = band_df.groupby('price_band', observed=True).apply(


,price_band,n_properties,MAPE,MdAPE
0,Under $500k,1810.0,0.255788,0.137167
1,$500k-$1M,5326.0,0.133024,0.088053
2,$1M-$2M,3945.0,0.141542,0.107719
3,Over $2M,1760.0,0.137711,0.111716


## 5. Save Metrics Summary
Saving the full metrics table to the outputs folder as metrics_summary.csv, 
this week's required deliverable.

In [7]:
output_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\outputs'
output_path = output_folder + '\\metrics_summary.csv'
metrics_summary.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\outputs\metrics_summary.csv


## Summary of Findings

**Full metrics (all models):**

| Model              | R²    | MAPE   | MdAPE  |
|---------------------|-------|--------|--------|
| Linear Regression   | 0.649 | 34.5%  | 22.2%  |
| Decision Tree        | 0.774 | 22.4%  | 14.9%  |
| Random Forest        | 0.876 | 15.1%  | 9.5%   |
| XGBoost               | 0.892 | 15.4%  | 10.2%  |

Interesting nuance: XGBoost has the best R² (explains the most overall price 
variance), but Random Forest has marginally better MAPE and MdAPE (typical 
percentage error). This suggests XGBoost handles harder-to-predict properties 
(likely higher-end homes) slightly better, boosting its variance-explained 
score, while Random Forest is marginally more consistent for a 
"typical"-priced home. Both are strong, defensible choices as the production 
model -- R² and MAPE aren't measuring quite the same thing.

**Price band analysis (XGBoost):**

| Price Band     | # Properties | MAPE   | MdAPE  |
|----------------|--------------|--------|--------|
| Under $500k    | 1,810        | 25.6%  | 13.7%  |
| $500k-$1M      | 5,326        | 13.3%  | 8.8%   |
| $1M-$2M        | 3,945        | 14.2%  | 10.8%  |
| Over $2M       | 1,760        | 13.8%  | 11.2%  |

Key insight: the model performs meaningfully worse on properties under 
$500k -- roughly double the error rate of every other price band. This makes 
sense for a few likely reasons: (1) lower-priced homes have less price 
variance to work with, so the same dollar-error translates into a much larger 
*percentage* error, (2) sub-$500k CA single-family homes are a smaller, more 
unusual segment of this dataset (only 1,810 of 12,841 test properties, ~14%), 
giving the model fewer examples to learn from, and (3) pricing at the low end 
may be driven more by factors outside our feature set (condition, deferred 
maintenance, distressed sales) that don't show up in structured MLS data. This 
is a natural limitation to flag for stakeholders rather than something to 
"fix" -- it's a real data/business constraint, not a modeling error.

**Recommendation:** XGBoost (or Random Forest) as the production model, with 
an explicit caveat that predictions for homes under $500k carry meaningfully 
higher uncertainty and should be flagged as lower-confidence in the app.